In [ ]:
# Imports
import numpy as np
import pandas as pd
import tensorflow as tf
from collections import Counter
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, LSTM, Dropout, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from prepare_data import clean_sentences, combine_reviews

In [3]:
# Load the CSV file
csv_data = 'data/tiny_data_reduced.csv' # Changed for each machine.
data = pd.read_csv(csv_data)

In [4]:
# Initialize the lists
positive_reviews = []
negative_reviews = []
neutral_reviews = []

# Clean the reviews and split them into positive, negative, and neutral
def review_splitting(row):
    cleaned_review = clean_sentences(pd.Series([row['1']]))[0]
    if row['0'] >= 8:
        positive_reviews.append(cleaned_review)
    elif row['0'] <= 4:
        negative_reviews.append(cleaned_review)
    else:
        neutral_reviews.append(cleaned_review)

# Applies review_splitting() to each row in the dataset
result = data.apply(review_splitting, axis = 1)

# Labels the reviews and convert to a numpy array
labels = [0] * len(negative_reviews) + [1] * len(neutral_reviews) + [2] * len(positive_reviews)
labels = np.array(labels)

combined_list = positive_reviews + neutral_reviews + negative_reviews

In [5]:
# Check the total number of words after sentence cleaning
total_words = []
for sentence in combined_list:
    words = sentence.split()
    for word in words:
        total_words.append(word)

# Check the number of unique words after sentence cleaning
unique_words = []
for sentence in combined_list:
    words = sentence.split()
    for word in words:
        if word not in unique_words:
            unique_words.append(word)

# Find the 95th percentile of the sequence lengths to not guess maxlen in the pad sequences
sequence_lengths = []
for review in combined_list:
    words = review.split()
    length = len(words)
    sequence_lengths.append(length)
percentile_95 = np.percentile(sequence_lengths, 95)

# Check how many words that appear more than 5 times. This is to better determine the size of num_words in the Tokenizer
data_count = Counter(total_words)
word_frequency = []
for word, count in data_count.items():
    if count >= 5:
        word_frequency.append(word)


print(f'Total words: {len(total_words)}')
print(f'Unique words: {len(unique_words)}')
print(f"95th Percentile Length: {percentile_95}")
print(f"Words that appear more than 5 times: {len(word_frequency)}")

Total words: 95516
Unique words: 7346
95th Percentile Length: 52.0
Words that appear more than 5 times: 1991


In [ ]:
# Tokenize the data
num_words = len(word_frequency)
tokenizer = Tokenizer(num_words = num_words)
tokenizer.fit_on_texts(combined_list)
sequences = tokenizer.texts_to_sequences(combined_list)

# Pad the sequences
maxlen = int(percentile_95)
pading = pad_sequences(sequences, maxlen = maxlen)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(pading, labels, test_size = 0.2, random_state = 42)

In [9]:
# Building the LSTM model
model = Sequential()
model.add(Embedding(input_dim = 2000, output_dim = 128, mask_zero = True))
model.add(LSTM(units = 128, return_sequences = False))
model.add(Dropout(0.5))
model.add(Dense(units = 3, activation = 'softmax'))

# LSTM - Vurder kernel_regularizer og recurrent_regularizer for å motvirke overfitting

In [ ]:
# Compiling the model
model.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

# Training the model
model.fit(X_train, y_train, epochs = 1, batch_size = 1024, validation_data = (X_val, y_val), verbose = 1)

# Save the model
model.save('./data/model/sentiment_lstm_model.keras')

print('Model training complete and saved as "./data/model/sentiment_lstm_model.keras"')

ValueError: Could not interpret metric identifier: f1